# Research 71 — RealMLP-TD validation across CV splits (comparison)

Compares the submitted top-pick recipe (0.8 × reference + 0.2 × VIP × region LightGBM) with RealMLP-TD alone and with a fixed 80/20 RealMLP blend. The comparison uses the same CV split for both models on each split, so each row is paired.

**Predeclared rule (set before any new split was run):** RealMLP-TD alone replaces the submitted recipe only if, on each of the three new split seeds (2025, 1337, 31415), its overall macro F1 is above the submitted recipe's and it is higher than that recipe on all three folds. That is 9 of 9 fold slices positive. Seed 42 is context only and does not decide. The fixed 80/20 RealMLP blend is secondary.

Uses saved OOF predictions only. Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)
COLS = [f"prob_{i}" for i in range(4)]


def load(name):
    frame = pd.read_csv(ROOT / "predictions" / name)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), name
    values = frame[COLS].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), name
    return values / values.sum(axis=1, keepdims=True)


def fold_index(seed):
    return list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )


p_ref = load("segment-oof.csv")
VIP = {
    42: "segment-vip-region-balanced-oof.csv",
    2025: "segment-vip-region-seed-2025-oof.csv",
    1337: "segment-vip-region-seed-1337-oof.csv",
    31415: "segment-vip-region-seed-31415-oof.csv",
}
REALMLP = {
    42: "segment-realmlp-td-oof.csv",
    2025: "segment-realmlp-td-seed2025-oof.csv",
    1337: "segment-realmlp-td-seed1337-oof.csv",
    31415: "segment-realmlp-td-seed31415-oof.csv",
}
NEW_SPLITS = [2025, 1337, 31415]

# Sanity check: the submitted recipe on split 42 must match notebook 28 (0.751552).
sub42 = 0.8 * p_ref + 0.2 * load(VIP[42])
assert abs(f1_score(y, sub42.argmax(1), average="macro") - 0.751552) < 1e-5
print("sanity ok: submitted recipe, split 42 = 0.751552")

sanity ok: submitted recipe, split 42 = 0.751552


## Reproducibility: local vs remote, split 42

In [2]:
local42 = load(REALMLP[42])
remote42 = load("segment-realmlp-td-seed42-remote-oof.csv")
repro = pd.DataFrame(
    [
        {
            "run": "local (notebook 65)",
            "macro_f1": f1_score(y, local42.argmax(1), average="macro"),
        },
        {
            "run": "Vast.ai (notebook 70)",
            "macro_f1": f1_score(y, remote42.argmax(1), average="macro"),
        },
    ]
)
display(repro.round(6))
print("max |prob diff| local vs remote:", float(np.abs(local42 - remote42).max()))
print("label agreement:", float((local42.argmax(1) == remote42.argmax(1)).mean()))

,run,macro_f1
0,local (notebook 65),0.754231
1,Vast.ai (notebook 70),0.754808


max |prob diff| local vs remote: 0.5760340310680028
label agreement: 0.9799711538461539


## Paired comparison on each split

In [3]:
rows = []
for seed in [42, *NEW_SPLITS]:
    folds = fold_index(seed)
    sub = 0.8 * p_ref + 0.2 * load(VIP[seed])
    rm = load(REALMLP[seed])
    blend = 0.8 * p_ref + 0.2 * rm
    sub_folds = [f1_score(y[v], sub[v].argmax(1), average="macro") for _, v in folds]
    row = {
        "split_seed": seed,
        "role": "context" if seed == 42 else "decides",
        "submitted_macro_f1": f1_score(y, sub.argmax(1), average="macro"),
        "realmlp_alone_macro_f1": f1_score(y, rm.argmax(1), average="macro"),
        "realmlp_blend_macro_f1": f1_score(y, blend.argmax(1), average="macro"),
        "submitted_weighted_f1": f1_score(y, sub.argmax(1), average="weighted"),
        "realmlp_alone_weighted_f1": f1_score(y, rm.argmax(1), average="weighted"),
    }
    for k, (_, v) in enumerate(folds, start=1):
        row[f"fold_{k}_delta_alone"] = (
            f1_score(y[v], rm[v].argmax(1), average="macro") - sub_folds[k - 1]
        )
    rows.append(row)
table = pd.DataFrame(rows)
table["overall_delta_alone"] = table["realmlp_alone_macro_f1"] - table["submitted_macro_f1"]
display(table.round(6))
table.to_csv(ROOT / "assets" / "segment-realmlp-td-split-validation.csv", index=False)

,split_seed,role,submitted_macro_f1,realmlp_alone_macro_f1,realmlp_blend_macro_f1,submitted_weighted_f1,realmlp_alone_weighted_f1,fold_1_delta_alone,fold_2_delta_alone,fold_3_delta_alone,overall_delta_alone
0,42,context,0.751552,0.754231,0.752264,0.765087,0.767629,0.002630,0.002985,0.002430,0.002679
1,2025,decides,0.751547,0.754713,0.752749,0.765136,0.768166,0.002846,0.002153,0.004494,0.003166
2,1337,decides,0.751967,0.753746,0.752520,0.765531,0.767107,0.001496,0.001813,0.002041,0.001778
3,31415,decides,0.751969,0.753761,0.752696,0.765502,0.767220,0.004123,-0.000293,0.001525,0.001792


## Verdict

In [4]:
decided = table[table["role"] == "decides"]
fold_cols = [f"fold_{k}_delta_alone" for k in (1, 2, 3)]
passes = (decided["overall_delta_alone"] > 0) & (decided[fold_cols] > 0).all(axis=1)
positive_slices = int((decided[fold_cols] > 0).sum().sum())
print("Fold slices positive (of 9):", positive_slices)
print("Splits passing (of 3):", int(passes.sum()))
print("Predeclared rule:", "VALIDATED" if bool(passes.all()) else "NOT VALIDATED")

Fold slices positive (of 9): 8
Splits passing (of 3): 2
Predeclared rule: NOT VALIDATED
